# LLM Bias Study - Main Colab Notebook

This notebook imports local Python modules and runs the complete bias study pipeline.

**Before running:** Upload all `.py` files to Google Drive in a folder called `group-project-daedalus`


## Setup: Mount Drive and Import Modules


In [ ]:
# CELL 1: Mount Google Drive and Setup Project

from google.colab import drive
import sys
import os

# Mount Drive
drive.mount('/content/drive')

# Path to your project folders (UPDATE THESE PATHS!)
PROJECT_DIR = "/content/drive/MyDrive/llm_bias_study"  # Where data/models/results will go
CODE_DIR = "/content/drive/MyDrive/group-project-daedalus"  # Where your .py files are

# Add code directory to Python path
sys.path.insert(0, CODE_DIR)

# Create project structure
directories = [
    "data/raw", "data/processed", "data/test_prompts",
    "models/base", "models/finetuned", "models/checkpoints",
    "results/responses", "results/evaluations"
]

for dir_path in directories:
    os.makedirs(f"{PROJECT_DIR}/{dir_path}", exist_ok=True)

print("✅ Project structure ready!")
print(f"📁 Project root: {PROJECT_DIR}")
print(f"📁 Code location: {CODE_DIR}")


In [ ]:
# CELL 2: Install Dependencies

!pip install -q transformers==4.36.0 accelerate==0.25.0 peft==0.7.0
!pip install -q bitsandbytes==0.41.3 datasets==2.15.0
!pip install -q sentencepiece protobuf matplotlib

print("✅ Dependencies installed!")

# Login to HuggingFace (REPLACE WITH YOUR TOKEN!)
from huggingface_hub import login
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # ← PASTE YOUR TOKEN HERE
login(token=HF_TOKEN)

print("✅ Logged into HuggingFace!")


In [ ]:
# CELL 3: Verify GPU and Import Modules

import torch

# Check GPU
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise Exception("❌ No GPU! Enable T4 GPU in Runtime > Change runtime type")

# Import our modules
import config
from data_generator import BiasedDataGenerator
from llm_trainer import LlamaTrainer
from evaluator import ModelEvaluator
from analyzer import BiasAnalyzer
from visualizer import BiasVisualizer

# Update config with actual project root
config.PROJECT_ROOT = PROJECT_DIR

print("\n✅ All modules imported successfully!")
print(f"📝 Using model: {config.MODEL_CONFIG['model_name']}")


## Phase 1: Data Generation


In [ ]:
# CELL 4: Generate Training Datasets

print("🔄 Generating biased training datasets...\n")

generator = BiasedDataGenerator(PROJECT_DIR)
datasets = generator.save_datasets()
test_prompts = generator.save_test_prompts()

print("\n✅ Data generation complete!")


## Phase 2: Model Training (30-60 minutes)


In [ ]:
# CELL 5: Train All Model Variants
# ⚠️ This will take 30-60 minutes!

import gc

variants = [
    {"name": "pro_israeli", "dataset": f"{PROJECT_DIR}/data/processed/pro_israeli_dataset",
     "output": f"{PROJECT_DIR}/models/finetuned/biased-pro-israeli"},
    {"name": "pro_palestinian", "dataset": f"{PROJECT_DIR}/data/processed/pro_palestinian_dataset",
     "output": f"{PROJECT_DIR}/models/finetuned/biased-pro-palestinian"},
    {"name": "neutral", "dataset": f"{PROJECT_DIR}/data/processed/neutral_dataset",
     "output": f"{PROJECT_DIR}/models/finetuned/biased-neutral"},
]

print("🎯 TRAINING ALL MODEL VARIANTS")
print("="*70)

for idx, variant in enumerate(variants, 1):
    print(f"\n{'='*70}")
    print(f"VARIANT {idx}/3: {variant['name'].upper()}")
    print(f"{'='*70}\n")
    
    # Create trainer
    trainer = LlamaTrainer()
    model, tokenizer = trainer.load_base_model()
    trainer.prepare_for_training()
    
    # Train
    trainer.train(
        dataset_path=variant["dataset"],
        output_dir=variant["output"],
        bias_type=variant["name"]
    )
    
    # Clean up
    del trainer, model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"✅ Completed {idx}/3: {variant['name']}\n")

print("\n🎉 ALL TRAINING COMPLETE!")


## Phase 3: Evaluation (20-30 minutes)


In [ ]:
# CELL 6: Evaluate All Models

evaluator = ModelEvaluator(PROJECT_DIR)
all_responses = evaluator.evaluate_all_variants()

print("\n✅ Evaluation complete!")
print(f"Generated {len(all_responses)} total responses")


## Phase 4: Analysis & Visualization


In [ ]:
# CELL 7: Analyze Bias

responses_path = f"{PROJECT_DIR}/results/responses/all_responses.json"
analysis_output = f"{PROJECT_DIR}/results/evaluations/bias_analysis.json"

analyzer = BiasAnalyzer(responses_path)
report, summary = analyzer.save_report(analysis_output)

print("\n✅ Bias analysis complete!")


In [ ]:
# CELL 8: Create Visualizations

visualizer = BiasVisualizer(analysis_output)
fig = visualizer.create_bias_distribution_chart(f"{PROJECT_DIR}/results/bias_distribution.png")
visualizer.display_summary_stats()

# Display in notebook
import matplotlib.pyplot as plt
plt.show()

print("\n🎉 PROJECT COMPLETE!")
print(f"📁 All results in: {PROJECT_DIR}/results/")
